# 性能内存与迭代

学习目标：先核对结果再比较有限规模的实际耗时，区分数组存储量与进程内存，使用 out、连续布局和分块控制计算成本。

前置知识：向量化、广播、通用函数、视图与副本、数组读写、函数与循环。

运行环境：Python 3.12、NumPy 2.5。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

首个代码单元导入 NumPy，后续单元沿用 np。计时只使用本机 CPU 和有限规模数组，实际耗时随运行条件变化；本章不要求安装选学工具。

标明“预期异常”的单元会直接显示原始报错；阅读异常类型与原因后，继续运行下一单元。

## 1 先确认计算结果

把一组整数读数按“乘 2，再加 1”换算。先分别用 Python 循环和数组表达式完成计算，再判断两种写法是否适合比较耗时。

下面两种实现接收同一数组，都新建结果数组并保留输入。比较时检查形状、dtype 和数值；array_equal 检查形状与元素是否相同，dtype 另行检查。本例使用范围很小的 int64 整数，没有溢出，适合精确比较。

In [1]:
import numpy as np


def scale_loop(values):
    result = np.empty_like(values)
    for index in range(values.size):
        result[index] = values[index] * 2 + 1
    return result


def scale_array(values):
    return values * 2 + 1


values = np.array([0, 2, 5, 9], dtype=np.int64)
loop_result = scale_loop(values)
array_result = scale_array(values)
print(loop_result, array_result)  # 预期：两个数组均为 [1 5 11 19]。
print(loop_result.shape, loop_result.dtype)  # 预期：(4,) int64。
print(np.array_equal(loop_result, array_result))
print(loop_result.dtype == array_result.dtype)
# 都得到 [1, 5, 11, 19]，形状为 (4,)，dtype 为 int64，两项检查均为 True。

[ 1  5 11 19] [ 1  5 11 19]
(4,) int64
True
True


## 2 重复计时与输入规模

timeit.repeat 返回多次测量的总耗时，单位是秒。repeat 指测量组数，number 指每组执行次数；总耗时除以 number 才是每次调用的平均耗时。globals 让计时语句访问本单元的变量和函数。

下面沿用 scale_loop 和 scale_array，分别比较 20 与 20000 个元素，每组运行 10 次，共测量 5 组。创建输入、核对结果和打印都在计时区外，函数调用与结果分配在计时区内。若断言不成立，程序会在计时前停止。

观察全部测量值，并把最小值作为当前条件下的耗时下界参考；不要把单次波动或固定加速倍数当成结论。timeit 默认暂时关闭循环垃圾回收，这也属于计时条件。

In [2]:
from timeit import repeat

number = 10
for size in (20, 20_000):
    values = np.arange(size, dtype=np.int64)
    expected = scale_loop(values)
    actual = scale_array(values)
    assert expected.dtype == actual.dtype
    assert np.array_equal(expected, actual)

    for label, statement in (
        ("循环", "scale_loop(values)"),
        ("数组", "scale_array(values)"),
    ):
        times = repeat(statement, globals=globals(), repeat=5, number=number)
        microseconds = np.array(times) / number * 1_000_000
        print(size, label, "微秒/次：", np.round(microseconds, 3))
        print("最小值：", round(float(microseconds.min()), 3))
# 打印两种规模、两种实现的真实测量值；没有预设的快慢倍数。

20 循环 微秒/次： [12.18 10.54 10.53 10.41 12.73]
最小值： 10.41
20 数组 微秒/次： [11.77  7.82  7.76  8.03  8.  ]
最小值： 7.76


20000 循环 微秒/次： [15349.96 17292.45 13957.07 13354.06 10944.92]
最小值： 10944.92
20000 数组 微秒/次： [58.7  43.84 43.25 53.63 44.55]
最小值： 43.25


## 3 把准备成本算清楚

计时前完成的工作不会进入测量结果；timeit 的 setup 参数也在计时区外执行。若实际任务每次都从列表创建数组，就应另测包含转换的完整调用。

下面沿用 scale_array 和 repeat，对照“已有数组”和“每次先转换列表”两种条件。输入列表的创建仍在计时区外，两种条件不能混称为同一任务的耗时。

In [3]:
source_list = list(range(20_000))
prepared = np.asarray(source_list, dtype=np.int64)
assert np.array_equal(scale_array(prepared), scale_array(np.asarray(source_list, dtype=np.int64)))

for label, statement in (
    ("已有数组", "scale_array(prepared)"),
    ("含列表转换", "scale_array(np.asarray(source_list, dtype=np.int64))"),
):
    times = repeat(statement, globals=globals(), repeat=5, number=10)
    print(label, "微秒/次：", np.round(np.array(times) / 10 * 1_000_000, 3))
# 输出各组实测耗时；第二种条件每次都包括创建 int64 数组的成本。

已有数组 微秒/次： [71.33 54.77 76.99 64.35 52.46]
含列表转换 微秒/次： [1160.37 1834.21 1242.15 1213.97 1485.33]


## 4 数组存储量与临时结果

### 4.1 nbytes 的范围

nbytes 统计数组元素对应的字节数，不包含数组对象的其他属性。视图可能共享同一缓冲区，所以把多个数组的 nbytes 相加，并不能得到独立分配的总量，更不能当成进程常驻内存（RSS）或运行峰值。

下面比较原数组、切片视图与独立副本。进程内存还包括解释器、库及其他对象；本章只核对数组元素字节数，不测量进程内存。

In [4]:
original = np.arange(12, dtype=np.int64)
view = original[::2]
copied = view.copy()

print(original.nbytes, view.nbytes, copied.nbytes)  # 预期：96 48 48，分别是各数组可见元素的字节数。
print(view.size * view.itemsize)
print(np.shares_memory(original, view), np.shares_memory(original, copied))
# 分别为 96、48、48 字节；view 的 48 字节来自 original，不能重复计为新缓冲区。
# 共享检查依次为 True、False。

96 48 48
48
True False


### 4.2 用 out 复用结果空间

values * 2 + 1 包含两次数组运算：乘法产生中间结果，加法产生最终结果。ufunc 的 out 可以指定已有数组保存结果。下面先把乘法写入 result，再在同一数组上加 1，输入保持不变。

out 的形状和类型须适合计算；本例统一使用 int64。它能减少这里显式创建的中间数组，但不代表 NumPy 内部绝不会申请临时空间；输入输出重叠时，ufunc 可能为处理数据依赖而复制。

In [5]:
values = np.arange(6, dtype=np.int64)
intermediate = values * 2
expected = intermediate + 1

result = np.empty_like(values)
np.multiply(values, 2, out=result)
np.add(result, 1, out=result)

print(result, values)  # 预期：result 为 [1 3 5 7 9 11]，values 仍为 [0 1 2 3 4 5]。
print(np.array_equal(result, expected), np.shares_memory(result, values))
print(intermediate.nbytes, result.nbytes)
# 结果为 [1, 3, 5, 7, 9, 11]，输入仍为 [0, 1, 2, 3, 4, 5]。
# 两个结果相等且 result 不与输入共享；一个长度为 6 的 int64 缓冲区为 48 字节。

[ 1  3  5  7  9 11] [0 1 2 3 4 5]
True False
48 48


如需比较重复计算，结果缓冲区是否预先分配也要写进条件。下面沿用 scale_array 与 repeat，比较每次创建结果和复用预分配结果；复用函数每次都从 values 重新计算，不会把上一次结果继续放大。

In [6]:
def scale_into(values, result):
    np.multiply(values, 2, out=result)
    np.add(result, 1, out=result)
    return result


values = np.arange(20_000, dtype=np.int64)
result = np.empty_like(values)
assert np.array_equal(scale_into(values, result), scale_array(values))

for label, statement in (
    ("每次分配", "scale_array(values)"),
    ("复用输出", "scale_into(values, result)"),
):
    times = repeat(statement, globals=globals(), repeat=5, number=10)
    print(label, "微秒/次：", np.round(np.array(times) / 10 * 1_000_000, 3))
# 输出实测耗时；复用方案的 result 分配在计时区外。

每次分配 微秒/次： [95.99 84.36 81.82 83.39 82.01]
复用输出 微秒/次： [33.31 31.27 28.23 30.68 28.21]


## 5 连续性与转换成本

C 连续和 F 连续分别表示数组按 C 或 Fortran 顺序连续存储，可以通过 flags 查看。带步长的切片可能两者都不是。ascontiguousarray 返回至少一维的 C 连续数组。对于本例二维输入，它保持形状和内容；已有数组满足要求时可直接返回它，否则需要复制。零维输入会变成形状为 (1,) 的一维数组，不能概括为总是保持形状。

先用小数组观察布局。不能仅凭“不连续”就判断某项运算必然更慢；是否值得复制，要连同转换成本一起测量。

In [7]:
base = np.arange(24, dtype=np.int64).reshape(4, 6)
strided = base[:, ::2]
contiguous = np.ascontiguousarray(strided)

print(strided)  # 预期：形状为 (4, 3)；逐行取出 0 至 22 的偶数。
print(strided.shape, strided.dtype)  # 预期：(4, 3) int64。
print(strided.flags.c_contiguous, strided.flags.f_contiguous)  # 预期：False False。
print(contiguous.flags.c_contiguous, np.array_equal(strided, contiguous))  # 预期：True True，连续化后数值不变。
print(np.shares_memory(base, strided), np.shares_memory(base, contiguous))
print(np.ascontiguousarray(contiguous) is contiguous)
# 切片形状为 (4, 3)、dtype 为 int64，既非 C 连续也非 F 连续。
# 副本内容相等且 C 连续；共享关系为 True、False；最后返回同一对象。

scalar = np.array(7)
scalar_contiguous = np.ascontiguousarray(scalar)
print(scalar.shape, scalar_contiguous.shape)  # () (1,)，零维输入提升为一维。
print(scalar_contiguous)  # [7]，数值保留。

[[ 0  2  4]
 [ 6  8 10]
 [12 14 16]
 [18 20 22]]
(4, 3) int64
False False
True True
True False
True
() (1,)
[7]


下面扩大到 (600, 200) 的有限切片，沿每行求和。沿用 repeat，分别测量直接求和、对已转换副本求和、每次转换后求和。第三种才包括每次复制的成本。

In [8]:
base = np.arange(600 * 400, dtype=np.int64).reshape(600, 400)
strided = base[:, ::2]
contiguous = np.ascontiguousarray(strided)
assert np.array_equal(np.sum(strided, axis=1), np.sum(contiguous, axis=1))

for label, statement in (
    ("直接求和", "np.sum(strided, axis=1)"),
    ("已转换副本", "np.sum(contiguous, axis=1)"),
    ("转换后求和", "np.sum(np.ascontiguousarray(strided), axis=1)"),
):
    times = repeat(statement, globals=globals(), repeat=5, number=10)
    print(label, "微秒/次：", np.round(np.array(times) / 10 * 1_000_000, 3))
# 输出三种条件的真实耗时；不能只拿“已转换副本”的耗时决定是否值得转换。

直接求和 微秒/次： [169.39 118.9  130.27  87.94  87.49]
已转换副本 微秒/次： [85.11 63.94 71.07 64.52 62.62]
转换后求和 微秒/次： [508.34 258.7  412.31 318.45 191.67]


## 6 广播规模与分块

广播可以省去显式重复输入，却仍会产生结果数组。若 N 个数与 M 个参考值分别求差，输出形状为 (N, M)，元素数为 N × M。N、M 分别表示输入数和参考值个数。

下面只计算假设规模的字节数，不实际创建大数组。float64 的每个元素占 8 字节。

In [9]:
count = 200_000
reference_count = 100_000
element_bytes = np.dtype(np.float64).itemsize
estimated_bytes = count * reference_count * element_bytes
print(estimated_bytes, estimated_bytes / 1_000_000_000)
# 仅一个差值结果就需要 160000000000 字节，即 160 GB（十进制）。
# 这里只做整数乘法，没有分配该数组；其他结果和输入还需额外空间。

160000000000 160.0


若只需要每个输入到最近参考值的平方差，就不必一直保存完整的两两结果。将输入分成小块，每块仍用数组运算，然后把每行最小值写入最终输出。

下面 9 个输入与 3 个参考值比较，每块最多 4 行。中间差值最多为 (4, 3)，而完整差值为 (9, 3)。分块限制中间数组大小，但最终 9 个结果仍需保存；它增加外层循环，是否更快要按任务测量。

In [10]:
points = np.arange(9, dtype=np.int64)
references = np.array([0, 4, 8], dtype=np.int64)
expected = np.min((points[:, np.newaxis] - references) ** 2, axis=1)
nearest = np.empty_like(points)
block_size = 4
largest_buffer = 0

for start in range(0, points.size, block_size):
    stop = min(start + block_size, points.size)
    differences = points[start:stop, np.newaxis] - references
    largest_buffer = max(largest_buffer, differences.nbytes)
    np.square(differences, out=differences)
    np.min(differences, axis=1, out=nearest[start:stop])

print(nearest, np.array_equal(nearest, expected))  # 预期：[0 1 4 1 0 1 4 1 0] True。
print("最大差值缓冲区字节数：", largest_buffer)
print("最终输出字节数：", nearest.nbytes)
# 结果为 [0, 1, 4, 1, 0, 1, 4, 1, 0]，与完整计算相同。
# 最大差值缓冲区为 96 字节，最终输出为 72 字节；这些数不是进程峰值。
# 9 不能被 4 整除，最后一块只处理 1 行。

[0 1 4 1 0 1 4 1 0] True
最大差值缓冲区字节数： 96
最终输出字节数： 72


为比较完整计算与分块计算的实际耗时，把上面的两种写法分别放入函数。使用 2000 个输入、100 个参考值，每块最多 128 行；先核对结果，再沿用 repeat 计时。两种函数都在每次调用中分配最终输出。

In [11]:
def nearest_full(points, references):
    # (点数, 1) 与 (参考数,) 广播，沿每行寻找最小平方距离。
    return np.min((points[:, np.newaxis] - references) ** 2, axis=1)


def nearest_blocked(points, references, block_size):
    # 输出只存每个点的最小距离，中间差值每次只创建一块。
    result = np.empty_like(points)
    for start in range(0, points.size, block_size):
        # 最后一块可能不足 block_size；切片始终停在实际点数处。
        stop = min(start + block_size, points.size)
        differences = points[start:stop, np.newaxis] - references
        # 原地平方复用差值缓冲区；行最小值直接写入对应输出切片。
        np.square(differences, out=differences)
        np.min(differences, axis=1, out=result[start:stop])
    return result


points = np.arange(2000, dtype=np.int64)
references = np.arange(0, 2000, 20, dtype=np.int64)
# 先确认两种算法完成相同任务，再比较耗时与中间数组规模。
full_result = nearest_full(points, references)
blocked_result = nearest_blocked(points, references, 128)
assert full_result.dtype == blocked_result.dtype
assert np.array_equal(full_result, blocked_result)

for label, statement in (
    ("完整计算", "nearest_full(points, references)"),
    ("分块计算", "nearest_blocked(points, references, 128)"),
):
    # 每轮运行 5 次，除以 5 后换算为单次微秒数。
    times = repeat(statement, globals=globals(), repeat=3, number=5)
    print(label, "微秒/次：", np.round(np.array(times) / 5 * 1_000_000, 3))  # 预期：每种方法各显示三次实测微秒数；数值和快慢关系随环境变化。
print("完整差值字节数：", points.size * references.size * points.itemsize)  # 预期：1600000 字节，来自完整差值数组的元素数乘以 itemsize。
print("单块差值上限字节数：", 128 * references.size * points.itemsize)
print("最终输出字节数：", blocked_result.nbytes)
# 实测耗时随环境变化；差值分别为 1600000、最多 102400 字节，输出为 16000 字节。
# 完整表达式还会产生平方结果，以上字节数只核算指定数组，不是进程峰值。

完整计算

微秒/次： [2022.88 2213.52 2232.6 ]
分块计算 微秒/次： [1193.46  924.1  1075.08]
完整差值字节数： 1600000
单块差值上限字节数： 102400
最终输出字节数： 16000


## 7 选学：逐元素迭代
ndindex 按形状生成索引元组，最后一轴变化最快，适合需要索引的逐项操作。nditer 访问数组元素，默认顺序尽量匹配内存布局；需要固定顺序时可指定 order="C" 或 "F"。

使用 nditer 的 Python 循环仍逐项运行循环体，不能仅凭接口名称认定它比已有数组表达式更快。

In [12]:
a = np.array([[1, 2, 3], [4, 5, 6]])
for index in np.ndindex(a.shape):
    print(index, a[index])
# 索引从 (0, 0) 到 (1, 2)，最后一轴先变化。

with np.nditer(a.T, order="C") as iterator:
    values = [int(value) for value in iterator]
print(values)
# 按转置后数组的 C 顺序访问，得到 [1, 4, 2, 5, 3, 6]。

(0, 0) 1
(0, 1) 2
(0, 2) 3
(1, 0) 4
(1, 1) 5
(1, 2) 6
[1, 4, 2, 5, 3, 6]


nditer 默认只读，写入需用 op_flags 指定 readwrite。使用 with 管理迭代器，退出时完成可能需要的缓冲区写回。external_loop 可以让循环体一次处理一段元素，减少逐项进入 Python 循环的次数。

In [13]:
a = np.arange(6, dtype=np.int64).reshape(2, 3)
with np.nditer(a, op_flags=["readwrite"]) as iterator:
    for value in iterator:
        value[...] = value + 1
print(a)
# 修改为 [[1, 2, 3], [4, 5, 6]]；value[...] 写入迭代器提供的数组位置。

with np.nditer(a, flags=["external_loop"]) as iterator:
    for chunk in iterator:
        print(chunk)
# 这个 C 连续小数组一次提供 [1, 2, 3, 4, 5, 6]；不承诺所有布局的块长相同。

[[1 2 3]
 [4 5 6]]
[1 2 3 4 5 6]


## 8 选学：滑动窗口
sliding_window_view 用共享数据的视图表示相邻窗口。长度为 6 的数组使用宽度为 3 的窗口，得到 (4, 3)：4 个窗口，每个含 3 个元素。窗口会重复引用同一内存位置，默认只读。

In [14]:
from numpy.lib.stride_tricks import sliding_window_view

signal = np.arange(6, dtype=np.int64)
windows = sliding_window_view(signal, 3)
print(windows, windows.shape)  # 预期：四行滑动窗口 [0,1,2] 至 [3,4,5]，形状为 (4, 3)。
print(np.shares_memory(signal, windows), windows.flags.writeable)
print(signal.nbytes, windows.nbytes)
# 窗口为 [0, 1, 2]、[1, 2, 3]、[2, 3, 4]、[3, 4, 5]。
# 共享且只读；nbytes 为 48、96，窗口的 96 不是新增数据缓冲区大小。

signal[2] = 20
print(windows)
# 同一原始元素出现在三个窗口中，三个位置随原数组一起变化。

# 预期 ValueError：sliding_window_view 默认返回只读窗口，不能直接写入。
windows[0, 0] = -1

[[0 1 2]
 [1 2 3]
 [2 3 4]
 [3 4 5]] (4, 3)
True False
48 96
[[ 0  1 20]
 [ 1 20  3]
 [20  3  4]
 [ 3  4  5]]


ValueError: assignment destination is read-only

创建视图省去复制，并不消除后续运算量。若对长度为 N 的输入逐窗口处理宽度为 W 的元素，朴素滑窗计算通常是 O(NW)；N 是输入长度，W 是窗口宽度。某些任务有 O(N) 的专门算法，因此大窗口不能只看视图是否省内存。

In [15]:
signal = np.arange(6, dtype=np.int64)
windows = sliding_window_view(signal, 3)
window_sums = np.sum(windows, axis=-1)
print(window_sums)
print("窗口内元素位置数：", windows.size)
# 得到 [3, 6, 9, 12]；计算涉及 4 × 3 个窗口内位置，部分位置引用相同元素。

[ 3  6  9 12]
窗口内元素位置数： 12


## 9 选学：线程与外部加速入口
### 9.1 线程分工

NumPy 的许多底层运算会释放 GIL，但不是所有操作都如此。共享数组的并发读写可能产生竞争；更容易管理的方式是每个任务独立拥有数组，或只读取共享输入并分别返回结果。需要并发修改时应另行设计锁等同步方式。

下面只演示两个线程读取不同切片。ThreadPoolExecutor 的 map 提交调用；with 结束时等待任务完成并释放线程池。这个小例子用于观察数据分工，不据此判断线程加速。

In [16]:
from concurrent.futures import ThreadPoolExecutor

shared = np.arange(6, dtype=np.int64)
shared.flags.writeable = False
parts = [shared[:3], shared[3:]]

with ThreadPoolExecutor(max_workers=2) as executor:
    totals = list(executor.map(np.sum, parts))

print(np.array(totals), shared)
print(parts[0].flags.writeable, parts[1].flags.writeable)
# 分别返回 3、12，输入未修改，两块均为只读；线程池在 with 结束时关闭。

[ 3 12] [0 1 2 3 4 5]
False False


### 9.2 BLAS 与其他工具

线性代数计算可能由 OpenBLAS、MKL 等 BLAS 后端执行，后端可能自行使用多线程。测量矩阵任务时，要记录实际后端和线程配置；线程控制方式取决于后端，不能把某个环境变量视为通用开关。threadpoolctl 是官方文档列出的控制工具之一。

进一步的工具按瓶颈选择：

| 工具 | 中文名称／含义 | 适合继续了解的条件 |
| --- | --- | --- |
| Numba | Python 即时编译器 | 数值循环难以写成简洁数组表达式；计时要区分首次编译和已编译调用 |
| CuPy | 面向 GPU 的数组库 | 已具备适合的 GPU 环境并准备迁移数组计算；GPU 异步执行需要相应的计时与同步方式 |

这些是后续阅读入口，本章不安装或运行它们，也不由 CPU 示例推断其加速倍数。

## 本章小结

（1）先核对形状、dtype 和数值，再测量有限规模输入；重复计时并说明计时区包含哪些工作。

（2）nbytes 只描述数组元素字节数；共享视图、临时数组和进程内存需要分别理解。

（3）out、连续副本和分块改变分配或访问方式；判断收益时同时考虑准备成本、最终输出和实际耗时。

（4）迭代器、滑动窗口与线程各有适用条件。省去复制、增加线程或换接口，都不能独自证明运行更快。

## 练习

（1）补全计时条件。沿用 scale_loop 和 scale_array，分别对 50、5000 个 int64 元素先核对结果，再用 repeat 测量。固定 repeat=3、number=5，报告每组的微秒/次，并说明创建输入和分配输出是否计入；不得只给“更快”结论。

In [17]:
sizes = (50, 5000)
# 在此逐个创建输入，核对 shape、dtype 和数值，再测量两种实现。
# 检查：打印全部测量值和计时条件；不扩大到无上限的输入规模。

（2）输入必须保留。将下面“乘 3，再减 2”的计算改成两次 ufunc 调用，共用一个独立输出数组。验证输入未改变，解释为什么直接把输入作为 out 不满足约束。

In [18]:
readings = np.array([2, 4, 6, 8], dtype=np.int64)
original = readings.copy()
expected = readings * 3 - 2
# 在此创建输出，使用 np.multiply 和 np.subtract 的 out。
# 检查：结果与 expected 相等，输入与 original 相等，输出不与输入共享。

（3）内存条件改变后选择方法。7 个输入与 3 个参考值求最近平方差，要求差值缓冲区最多容纳 2 行。完成分块计算并与小规模完整计算核对；说明为何不能一次保存完整差值，最终输出是否仍需内存。

In [19]:
points = np.arange(7, dtype=np.int64)
references = np.array([0, 3, 6], dtype=np.int64)
expected = np.min((points[:, np.newaxis] - references) ** 2, axis=1)
block_size = 2
# 在此分块计算，打印结果和最大差值缓冲区 nbytes。
# 检查：最后一块只有 1 行；每块差值不超过 2 × 3 × 8 字节。
# 写出方法选择理由，并区别该字节数、最终输出字节数与进程峰值。

（4）是否值得转换布局？下面的切片只需要沿行求和一次。先核对直接求和与转换后求和的结果，再测量两种完整调用并提出选择理由。如果改为重复使用已转换数组 100 次，计时条件应怎样调整？

In [20]:
base = np.arange(200 * 120, dtype=np.int64).reshape(200, 120)
strided = base[:, ::2]
# 在此核对结果，再比较直接求和与每次转换后求和。
# 检查：转换成本应计入一次性任务；重复使用时应说明转换发生几次。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| NumPy 官方文档（2.5） | [ufuncs](https://numpy.org/doc/2.5/reference/ufuncs.html#optional-keyword-arguments) 的 Optional keyword arguments：out、默认输出分配及重叠时的临时复制；[array_equal](https://numpy.org/doc/2.5/reference/generated/numpy.array_equal.html) 的定义：结果核对；[ndarray.nbytes](https://numpy.org/doc/2.5/reference/generated/numpy.ndarray.nbytes.html) 的定义与 Notes：元素字节数的范围；[shares_memory](https://numpy.org/doc/2.5/reference/generated/numpy.shares_memory.html) 的定义：共享检查；[ndarray.flags](https://numpy.org/doc/2.5/reference/generated/numpy.ndarray.flags.html) 的 Attributes、Notes 与 [ascontiguousarray](https://numpy.org/doc/2.5/reference/generated/numpy.ascontiguousarray.html) 的 Returns、Notes、Examples：布局、只读、转换及至少一维的返回条件；[Broadcasting](https://numpy.org/doc/2.5/user/basics.broadcasting.html#practical-example-vector-quantization) 的 Practical Example 尾部：大中间数组及外层循环；[sum](https://numpy.org/doc/2.5/reference/generated/numpy.sum.html) 与 [min](https://numpy.org/doc/2.5/reference/generated/numpy.min.html) 的 Parameters：按轴计算、out；[ndindex](https://numpy.org/doc/2.5/reference/generated/numpy.ndindex.html) 的定义与 [Iterating over arrays](https://numpy.org/doc/2.5/reference/arrays.nditer.html) 的 Single array iteration、Modifying array values、Using an external loop：迭代顺序、写回与分段；[sliding_window_view](https://numpy.org/doc/2.5/reference/generated/numpy.lib.stride_tricks.sliding_window_view.html) 的 Parameters、Notes、Examples：重叠、默认只读、形状及复杂度；[Thread Safety](https://numpy.org/doc/2.5/reference/thread_safety.html) 的共享读写说明、[Global Configuration Options](https://numpy.org/doc/2.5/reference/global_state.html#number-of-threads-used-for-linear-algebra) 的 Number of threads used for linear algebra：GIL、BLAS 后端与线程控制。 |
| Python 官方文档（3.12） | [timeit](https://docs.python.org/3.12/library/timeit.html) 的 Python Interface、Timer.timeit、Timer.repeat：重复次数、单位、setup、垃圾回收与最小值；[concurrent.futures](https://docs.python.org/3.12/library/concurrent.futures.html#executor-objects) 的 Executor Objects、ThreadPoolExecutor：map 与上下文管理器关闭。 |
| Numba 官方文档 | [A ~5 minute guide to Numba](https://numba.readthedocs.io/en/stable/user/5minguide.html) 的简介、Will Numba work for my code、How to measure the performance of Numba：数值循环及首次编译的计时区别，仅作为选学入口。 |
| CuPy 官方文档 | [Overview](https://docs.cupy.dev/en/stable/overview.html) 的 GPU 数组用途；[Performance Best Practices](https://docs.cupy.dev/en/stable/user_guide/performance.html#benchmarking) 的 Benchmarking：GPU 异步执行与相应计时工具，仅作为选学入口。 |